# Ejemplos didácticos de Bagging y Boosting con Breast Cancer

Este notebook usa el dataset Breast Cancer Wisconsin Diagnostic, disponible en `scikit-learn`, para ilustrar cómo evolucionan varios métodos de Bagging y Boosting al cambiar el número de estimadores. Es un ejemplo didáctico independiente de la comparativa principal.


## Librerías


In [ ]:
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


## Carga de datos y clase positiva

La clase positiva se define como diagnóstico maligno para calcular el AUC y el índice de Gini derivado del AUC. Este índice no debe confundirse con la impureza de Gini usada en CART.


In [ ]:
datos = load_breast_cancer(as_frame=True)
X = datos.data
y_original = datos.target

# scikit-learn codifica 0 = malignant y 1 = benign. Se transforma para que 1 sea malignant.
y = (y_original == 0).astype(int)

X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42,
)

resumen_datos = pd.DataFrame([{
    "filas": X.shape[0],
    "variables": X.shape[1],
    "train": X_entrenamiento.shape[0],
    "test": X_prueba.shape[0],
    "clase positiva": "malignant",
}])

resumen_datos


## Métrica

Se usa el índice de Gini derivado del AUC: `Gini = 2 * AUC - 1`. En las tablas se expresa en porcentaje para facilitar la lectura.


In [ ]:
def gini_porcentaje_desde_auc(y_real, puntuaciones):
    auc = roc_auc_score(y_real, puntuaciones)
    return 100.0 * (2.0 * auc - 1.0)


def evaluar_modelo(modelo, nombre_modelo, numero_estimadores=None):
    inicio = perf_counter()
    modelo.fit(X_entrenamiento, y_entrenamiento)
    tiempo_ejecucion = perf_counter() - inicio

    puntuaciones_entrenamiento = modelo.predict_proba(X_entrenamiento)[:, 1]
    puntuaciones_prueba = modelo.predict_proba(X_prueba)[:, 1]

    gini_entrenamiento = gini_porcentaje_desde_auc(y_entrenamiento, puntuaciones_entrenamiento)
    gini_prueba = gini_porcentaje_desde_auc(y_prueba, puntuaciones_prueba)

    return {
        "Modelo": nombre_modelo,
        "Número de estimadores": numero_estimadores,
        "Tiempo de ejecución": tiempo_ejecucion,
        "Gini en entrenamiento": gini_entrenamiento,
        "Gini en prueba": gini_prueba,
        "Diferencia": gini_entrenamiento - gini_prueba,
    }


def guardar_grafico_lineas(datos, columna_x, columna_y, titulo, etiqueta_x, etiqueta_y, ruta_salida):
    fig, ax = plt.subplots(figsize=(9, 5))
    for nombre_modelo, grupo in datos.groupby("Modelo"):
        grupo = grupo.sort_values(columna_x)
        ax.plot(grupo[columna_x], grupo[columna_y], marker="o", linewidth=1.8, label=nombre_modelo)
    ax.set_xscale("log")
    ax.set_xlabel(etiqueta_x)
    ax.set_ylabel(etiqueta_y)
    ax.set_title(titulo)
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend()
    fig.tight_layout()
    fig.savefig(ruta_salida, dpi=300)
    plt.show()


## Bagging


In [ ]:
numero_arboles = [1, 3, 5, 10, 15, 20, 30, 50, 100, 200, 500, 1000]
carpeta_tablas = Path("outputs/tables/ejemplos_didacticos")
carpeta_figuras = Path("outputs/figures/ejemplos_didacticos")
carpeta_tablas.mkdir(parents=True, exist_ok=True)
carpeta_figuras.mkdir(parents=True, exist_ok=True)

referencia_cart = pd.DataFrame([
    evaluar_modelo(DecisionTreeClassifier(random_state=42), "CART individual")
])
referencia_cart.to_csv(carpeta_tablas / "bagging_cart_referencia.csv", index=False)

filas_bagging = []
for n in numero_arboles:
    filas_bagging.append(evaluar_modelo(
        BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42), n_estimators=n, random_state=42, n_jobs=1),
        "Bagging CART",
        n,
    ))
    filas_bagging.append(evaluar_modelo(
        RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=1),
        "Random Forest",
        n,
    ))
    filas_bagging.append(evaluar_modelo(
        ExtraTreesClassifier(n_estimators=n, random_state=42, n_jobs=1),
        "Extra Trees",
        n,
    ))

resultados_bagging = pd.DataFrame(filas_bagging)
resultados_bagging.to_csv(carpeta_tablas / "bagging_resultados_por_estimadores.csv", index=False)

guardar_grafico_lineas(resultados_bagging, "Número de estimadores", "Gini en prueba", "Evolución del Gini en prueba - Bagging", "Número de árboles", "Gini en prueba (%)", carpeta_figuras / "bagging_convergencia_gini.png")
guardar_grafico_lineas(resultados_bagging, "Número de estimadores", "Diferencia", "Diferencia entre entrenamiento y prueba - Bagging", "Número de árboles", "Diferencia de Gini", carpeta_figuras / "bagging_delta_gini.png")
guardar_grafico_lineas(resultados_bagging, "Número de estimadores", "Tiempo de ejecución", "Tiempo de ejecución - Bagging", "Número de árboles", "Tiempo de ejecución (s)", carpeta_figuras / "bagging_tiempo_ejecucion.png")

resultados_bagging.head(9)


## Boosting


In [ ]:
filas_boosting = []
for n in numero_arboles:
    modelos = [
        ("AdaBoost", AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=n, random_state=42)),
        ("Gradient Boosting", GradientBoostingClassifier(n_estimators=n, random_state=42)),
        ("XGBoost", XGBClassifier(n_estimators=n, max_depth=2, learning_rate=0.1, objective="binary:logistic", eval_metric="auc", random_state=42, n_jobs=1, verbosity=0)),
        ("LightGBM", LGBMClassifier(n_estimators=n, learning_rate=0.1, random_state=42, verbose=-1)),
        ("CatBoost", CatBoostClassifier(iterations=n, learning_rate=0.1, depth=2, loss_function="Logloss", eval_metric="AUC", random_seed=42, verbose=False, allow_writing_files=False)),
    ]
    for nombre, modelo in modelos:
        filas_boosting.append(evaluar_modelo(modelo, nombre, n))

resultados_boosting = pd.DataFrame(filas_boosting)
resultados_boosting.to_csv(carpeta_tablas / "boosting_resultados_por_estimadores.csv", index=False)

guardar_grafico_lineas(resultados_boosting, "Número de estimadores", "Gini en prueba", "Evolución del Gini en prueba - Boosting", "Número de estimadores", "Gini en prueba (%)", carpeta_figuras / "boosting_convergencia_gini.png")
guardar_grafico_lineas(resultados_boosting, "Número de estimadores", "Diferencia", "Diferencia entre entrenamiento y prueba - Boosting", "Número de estimadores", "Diferencia de Gini", carpeta_figuras / "boosting_delta_gini.png")
guardar_grafico_lineas(resultados_boosting, "Número de estimadores", "Tiempo de ejecución", "Tiempo de ejecución - Boosting", "Número de estimadores", "Tiempo de ejecución (s)", carpeta_figuras / "boosting_tiempo_ejecucion.png")

resultados_boosting.head(10)


## Lectura breve

Las curvas permiten comparar de forma visual la evolución de los métodos. El ejemplo no establece conclusiones generales del TFG; solo ilustra diferencias de comportamiento en un dataset clásico.
